# Entropy market filter - Dynamic Top-10 S&P 500 allocation

Self-contained notebook (Colab or local). Shannon entropy is used **only** as a
market-regime classifier (Bullish / Neutral / Bearish); the expected-return input is the
plain rolling mean. The allocation solves a daily mean-variance SLSQP in the
Bullish/Neutral regimes and holds equal weight in the Bearish regime, with the maximum
per-name weight read directly from the regime.

**What is simplified vs the full project**
- **Universe:** only the *true* top-10 by market cap at each date are tradable
  (monthly membership, forward-filled to daily). E.g. NVDA is not tradable in 2001.
- **Allocation:** bounds come directly from the regime — **min 0**, **max 30% (Bullish) /
  15% (Neutral) / 10% (Bearish)**. No expanding bound calibration. SLSQP runs only in
  Bullish/Neutral; the Bearish gear is an explicit equal-weight (1/N) book.
- **Benchmarks:** a naive max-Sharpe top-10, equal-weight top-10, **and** S&P 500
  total-return buy-and-hold (no rebalancing).

**Inputs (provided, beside the notebook):** `dataset.csv` (top-10 price panel),
`sp500_top10_universe.csv` (monthly top-10 membership), `sp500_total_return.csv`
(^SP500TR). The notebook does **not** download or build the price dataset.

**Outputs:** `output/img/*.png` and `output/csv/*.csv`. The Overleaf-ready report
package is provided separately in the repository `report/` folder.

**Professor's rules kept:** `shannon_entropy` and the allocation loop are hand-coded
(only SLSQP is delegated to `scipy.optimize.minimize`); Big-O annotations are retained.

**Run:** execute every cell top to bottom. On Colab, upload the three CSVs next to the notebook.

## 1. Setup and imports

In [ ]:
"""Imports and small ticker/frequency helpers. Complexity: O(1)."""
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import math
import warnings

import numpy as np
import pandas as pd

import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

from scipy import stats
from scipy.optimize import minimize

warnings.filterwarnings("ignore", category=FutureWarning)


def compatible_offset(preferred: str, fallback: str) -> str:
    """
    Return the first pandas frequency alias supported by the installed version.

    Complexity: O(1).
    """
    try:
        pd.tseries.frequencies.to_offset(preferred)
        return preferred
    except ValueError:
        return fallback


def display_ticker(raw: str) -> str:
    """Human-readable ticker label (e.g. BRK-B -> BRK.B). Complexity: O(1)."""
    return str(raw).replace("-", ".")


def normalize_ticker(raw: object) -> str:
    """
    Canonical ticker key used to match universe tickers to price columns.

    Complexity: O(L), where L is the ticker string length (small constant).
    """
    return str(raw).strip().upper().replace(".", "-")


MONTH_END_FREQ = compatible_offset("ME", "M")

print("Libraries loaded. pandas", pd.__version__, "| numpy", np.__version__)


## 2. Configuration

The regime no longer selects an *objective*; it selects the **maximum weight** directly.
`REGIME_TO_MAX_WEIGHT` is the whole optimizer policy. The minimum weight is always 0.

**Data location (Colab-friendly).** The three input CSVs are read from a single
editable path variable, **`DATA_DIR`**, set in the cell below. Change that one line
to run on another machine or in Google Colab: for example, upload the CSVs and keep
`DATA_DIR = Path.cwd()`, or mount Google Drive and set
`DATA_DIR = Path("/content/drive/MyDrive/entropy")`. Outputs are always written
under `output/` in the working directory.

In [ ]:
MARKET_PHASE_ORDER = ["Bullish", "Neutral", "Bearish"]
MARKET_PHASE_COLORS = {"Bullish": "#2ca25f", "Neutral": "#969696", "Bearish": "#de2d26"}
# The entire optimizer policy: regime -> maximum per-name weight (min is always 0).
REGIME_TO_MAX_WEIGHT = {"Bullish": 0.30, "Neutral": 0.15, "Bearish": 0.10}
CRISIS_WINDOWS = [
    ("Global financial crisis", "2007-10-09", "2009-03-09"),
    ("Euro debt and US downgrade", "2011-07-01", "2011-12-31"),
    ("COVID crash", "2020-02-19", "2020-04-30"),
    ("Inflation and rates bear market", "2022-01-03", "2022-10-14"),
]


@dataclass
class EntropyConfig:
    """Simplified configuration. Complexity: O(1)."""
    root: Path
    dataset_filename: str = "dataset.csv"
    universe_filename: str = "sp500_top10_universe.csv"
    sp500_filename: str = "sp500_total_return.csv"
    output_root: Path | None = None
    lookback_days: int = 126
    min_periods: int = 90
    entropy_bins: int = 8
    top_n: int = 10
    max_weight: float = 0.30          # used only as the weight-heatmap colour scale
    risk_aversion: float = 4.0
    trading_days: int = 252
    warmup_start: str = "2001-01-01"
    analysis_start: str = "2002-01-31"
    analysis_end: str = "2025-12-31"
    phase_lookback_days: int = 126
    phase_entropy_window: int = 504
    phase_entropy_min_periods: int = 126
    phase_entropy_lower_quantile: float = 0.33
    phase_entropy_upper_quantile: float = 0.67

    def out(self, *parts: str) -> Path:
        base = self.output_root if self.output_root is not None else self.root
        return base.joinpath(*parts)

    @property
    def output_dir(self) -> Path:
        return self.out("output", "csv")

    @property
    def img_dir(self) -> Path:
        return self.out("output", "img")

    def _resolve(self, name: str) -> Path:
        p = Path(name)
        return p if p.is_absolute() else self.root / p

    @property
    def dataset_path(self) -> Path:
        return self._resolve(self.dataset_filename)

    @property
    def universe_path(self) -> Path:
        return self._resolve(self.universe_filename)

    @property
    def sp500_path(self) -> Path:
        return self._resolve(self.sp500_filename)


def ensure_dirs(cfg: EntropyConfig) -> None:
    """Create the output csv/ and img/ folders if missing. O(1)."""
    cfg.output_dir.mkdir(parents=True, exist_ok=True)
    cfg.img_dir.mkdir(parents=True, exist_ok=True)


# --- Data location -------------------------------------------------------
# DATA_DIR holds the three input CSVs. It supports running from this folder,
# from the repository root, or from a Colab directory containing the CSVs.
# Edit THIS ONE LINE to point elsewhere, e.g.:
#   DATA_DIR = Path("/content/drive/MyDrive/entropy")  # Colab + Google Drive
# Outputs are always written under the current working directory (ROOT).
DATA_CANDIDATES = [Path.cwd(), Path.cwd() / "progetto", Path.cwd().parent]
DATA_DIR = next((path for path in DATA_CANDIDATES if (path / "dataset.csv").exists()), Path.cwd())
ROOT = Path.cwd()
cfg = EntropyConfig(
    root=ROOT,
    dataset_filename=str(DATA_DIR / "dataset.csv"),
    universe_filename=str(DATA_DIR / "sp500_top10_universe.csv"),
    sp500_filename=str(DATA_DIR / "sp500_total_return.csv"),
)
ensure_dirs(cfg)
print("Data dir:  ", DATA_DIR)
print("Root:      ", ROOT)
print("Dataset:   ", cfg.dataset_path)
print("Universe:  ", cfg.universe_path)
print("S&P 500:   ", cfg.sp500_path)
print("Out CSV:   ", cfg.output_dir)
print("Out IMG:   ", cfg.img_dir)
print("Regime caps:", REGIME_TO_MAX_WEIGHT)


## 3. Dataset and the true top-10 universe

`dataset.csv` is a daily price panel of every ticker that was ever in the top-10.
A stock is **tradable on day _t_ only if it belongs to the top-10 of the most recent
month-end on/before _t_** (causal monthly→daily forward-fill) **and** has a price that
day. This is what restricts trading to the genuine top-10 (no NVDA in 2001).
Returns are *simple* returns.

In [ ]:
"""Dataset loading and the causal top-10 universe. All stages are O(T*N)."""

def read_price_panel(path: Path) -> pd.DataFrame:
    """Load the price panel CSV into a clean, de-duplicated, numeric DataFrame.
    In: csv path. Out: prices (index Date, columns normalized tickers). O(T*N)."""
    raw = pd.read_csv(path)
    date_col = "Date" if "Date" in raw.columns else raw.columns[0]
    raw[date_col] = pd.to_datetime(raw[date_col], errors="coerce")
    raw = raw.dropna(subset=[date_col]).set_index(date_col).sort_index()
    raw.index.name = "Date"
    raw = raw[~raw.index.duplicated(keep="last")]

    prices = raw.apply(pd.to_numeric, errors="coerce")
    prices = prices.replace([np.inf, -np.inf], np.nan)
    prices = prices.dropna(axis=1, how="all")
    prices.columns = [normalize_ticker(column) for column in prices.columns]
    prices = prices.loc[:, ~pd.Index(prices.columns).duplicated(keep="first")]
    return prices


def build_active_universe(cfg: EntropyConfig, returns: pd.DataFrame) -> pd.DataFrame:
    """Boolean Active[t, ticker]: ticker is in the top-10 as of the most recent
    month-end on/before t (causal) AND has a finite return at t.
    In: cfg, returns. Out: T*N bool DataFrame. O(T*N)."""
    u = pd.read_csv(cfg.universe_path)
    date_col = "Date" if "Date" in u.columns else u.columns[0]
    members_col = "Top 10 tickers" if "Top 10 tickers" in u.columns else u.columns[1]
    u[date_col] = pd.to_datetime(u[date_col])
    u = u.sort_values(date_col)
    u["members"] = u[members_col].apply(
        lambda s: frozenset(normalize_ticker(t) for t in str(s).split(",") if str(t).strip())
    )
    daily = pd.merge_asof(
        pd.DataFrame({"Date": returns.index}).sort_values("Date"),
        u[[date_col, "members"]].rename(columns={date_col: "Date"}),
        on="Date", direction="backward",
    ).set_index("Date")["members"]
    # Cache one precomputed boolean row per distinct monthly top-10 set, so the
    # dense T*N mask is filled without a per-date .loc scatter.
    cols = list(returns.columns)
    col_pos = {c: j for j, c in enumerate(cols)}
    members_by_day = daily.reindex(returns.index).to_numpy()
    mask = np.zeros((len(returns.index), len(cols)), dtype=bool)
    row_cache: dict = {}
    for i, members in enumerate(members_by_day):
        if isinstance(members, frozenset) and members:
            row = row_cache.get(members)
            if row is None:
                row = np.zeros(len(cols), dtype=bool)
                for c in members:
                    j = col_pos.get(c)
                    if j is not None:
                        row[j] = True
                row_cache[members] = row
            mask[i] = row
    active = pd.DataFrame(mask, index=returns.index, columns=cols) & returns.notna()
    return active


def prepare_dataset(cfg: EntropyConfig) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Load prices, compute simple returns, build the active-universe mask, and
    export the basic data CSVs. Out: (prices, returns, active). O(T*N)."""
    prices = read_price_panel(cfg.dataset_path)
    prices = prices.loc[
        (prices.index >= pd.Timestamp(cfg.warmup_start))
        & (prices.index <= pd.Timestamp(cfg.analysis_end))
    ]
    returns = prices.pct_change(fill_method=None).replace([np.inf, -np.inf], np.nan)
    returns = returns.dropna(how="all")
    active = build_active_universe(cfg, returns)

    prices.to_csv(cfg.output_dir / "daily_prices.csv", index_label="Date")
    returns.to_csv(cfg.output_dir / "daily_simple_returns.csv", index_label="Date")
    active.astype(int).to_csv(cfg.output_dir / "active_universe_daily.csv", index_label="Date")

    summary = pd.DataFrame(
        [
            ["Dataset path", str(cfg.dataset_path)],
            ["Price start", str(prices.index.min().date())],
            ["Price end", str(prices.index.max().date())],
            ["Price rows", str(len(prices))],
            ["Price columns", str(prices.shape[1])],
            ["Return rows", str(len(returns))],
            ["Mean active names", f"{active.sum(axis=1).mean():.2f}"],
            ["Min active names", str(int(active.sum(axis=1).min()))],
            ["Max active names", str(int(active.sum(axis=1).max()))],
        ],
        columns=["Item", "Value"],
    )
    summary.to_csv(cfg.output_dir / "dataset_summary.csv", index=False)
    return prices, returns, active


prices, returns, active = prepare_dataset(cfg)
print(f"Prices:  {prices.shape[0]} rows x {prices.shape[1]} columns")
print(f"Active names per day: mean={active.sum(axis=1).mean():.2f} min={int(active.sum(axis=1).min())} max={int(active.sum(axis=1).max())}")
prices.head()


## 4. Toy Shannon-entropy example

`shannon_entropy` is hand-coded from scratch (professor's requirement). Normalized to
[0,1]: 0 = deterministic, 1 = uniform/maximum disorder. Complexity O(n + B).

In [ ]:
def shannon_entropy(values: np.ndarray, bins: int = 8) -> float:
    """Normalized empirical Shannon entropy of one return window, in [0, 1].
    The window is binned into `bins` equal-width buckets over its own [min, max]
    (so it measures the SHAPE/spread of returns and is scale-invariant): 0 = one
    bucket (ordered), 1 = spread evenly (disordered).
    In: 1-D array. Out: float in [0, 1] (NaN if < 2 finite points). O(W + B)."""
    clean = np.asarray(values, dtype=float)
    clean = clean[np.isfinite(clean)]
    if clean.size < 2:                          # too few points -> entropy undefined
        return np.nan
    counts, _ = np.histogram(clean, bins=bins)  # O(W): scan [min,max] + assign buckets
    probs = counts[counts > 0] / clean.size     # O(B): counts -> probabilities p_b
    entropy = -np.sum(probs * np.log2(probs))   # O(B): H = -sum p_b log2 p_b
    return float(entropy / np.log2(bins))       # O(1): rescale to [0, 1] by log2(bins)


def toy_entropy_example(cfg: EntropyConfig) -> pd.DataFrame:
    """
    Generate and save the toy Shannon-entropy example.
    Complexity: O(B * n) where n is the example length.
    """
    rng = np.random.default_rng(7)
    examples = {
        "Structured trend": np.array([-0.002, -0.001, 0.000, 0.001, 0.002] * 12),
        "Two-state regime": np.array([-0.012, 0.012] * 30),
        "Noisy regime": rng.normal(0.0, 0.012, 60),
    }
    rows = []
    fig, ax = plt.subplots(figsize=(8, 4.8))
    for name, values in examples.items():
        h = shannon_entropy(values, cfg.entropy_bins)
        rows.append(
            {
                "Example": name,
                "Observations": len(values),
                "NormalizedEntropy": h,
                "StructureScore": 1.0 - h,
                "MeanReturn": float(np.mean(values)),
            }
        )
        counts, edges = np.histogram(values, bins=cfg.entropy_bins)
        centers = 0.5 * (edges[1:] + edges[:-1])
        ax.plot(centers, counts, marker="o", linewidth=1.5, label=f"{name}: H={h:.2f}")
    ax.set_title("Toy Shannon entropy examples")
    ax.set_xlabel("Return bin")
    ax.set_ylabel("Frequency")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.25)
    fig.tight_layout()
    fig.savefig(cfg.img_dir / "toy_entropy_example.png", dpi=180)
    plt.show()

    toy = pd.DataFrame(rows)
    toy.to_csv(cfg.output_dir / "toy_entropy_example.csv", index=False)
    return toy


def toy_allocation_example(cfg: EntropyConfig) -> dict:
    """Reproduce the report's 4-name mini-allocation (A,B,C,D) end to end and plot it:
    per-name entropy and rolling mean (inputs), then the Bullish SLSQP weights against
    the Bearish equal-weight book (outputs). Self-contained (inlines the same SLSQP QP).
    In: cfg. Out: dict of computed H / mu / weights. Saves toy_allocation.png. O(1)."""
    rets = {
        "A": [0.6, 0.8, 0.5, 0.7, 0.9, 0.6, 0.8, 0.7],
        "B": [2.0, -1.8, 1.5, -2.2, 0.3, 1.9, -1.0, 0.8],
        "C": [1.2, -0.3, 0.9, 1.5, -0.6, 1.1, 0.4, 0.8],
        "D": [-0.2, -0.5, 0.1, -0.4, -0.3, 0.0, -0.6, 0.2],
    }
    names = list(rets)
    n = len(names)
    R = np.array([rets[k] for k in names]) / 100.0          # percent -> decimal
    H = np.array([shannon_entropy(R[i], cfg.entropy_bins) for i in range(n)])
    mu = R.mean(axis=1)
    cov = 0.5 * (np.cov(R) + np.cov(R).T) + np.eye(n) * 1e-8  # same ridge as the real solver
    cap = REGIME_TO_MAX_WEIGHT["Bullish"]                    # Bullish toy -> 30% cap
    res = minimize(lambda w: 0.5 * cfg.risk_aversion * (w @ cov @ w) - mu @ w,
                   np.full(n, 1.0 / n), method="SLSQP",
                   jac=lambda w: cfg.risk_aversion * (cov @ w) - mu,
                   bounds=[(0.0, cap)] * n,
                   constraints=[{"type": "eq", "fun": lambda w: w.sum() - 1.0,
                                 "jac": lambda w: np.ones_like(w)}],
                   options={"ftol": 1e-10, "maxiter": 120})
    w_bull = res.x
    w_bear = np.full(n, 1.0 / n)                             # Bearish -> equal weight

    x = np.arange(n)
    fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.3))
    ax = axes[0]
    ax.bar(x, mu * 100.0, 0.5, color="tab:blue", label=r"rolling mean $\bar r_i$ (%)")
    ax.axhline(0.0, color="black", linewidth=0.8)
    ax.set_xticks(x); ax.set_xticklabels(names)
    ax.set_ylabel("Rolling mean (%)")
    ax.set_title("Inputs: expected return and entropy")
    axH = ax.twinx()
    axH.plot(x, H, "o-", color="tab:red", label=r"entropy $H_i$")
    axH.set_ylabel("Normalized entropy $H_i$"); axH.set_ylim(0.0, 1.0)
    h1, l1 = ax.get_legend_handles_labels(); h2, l2 = axH.get_legend_handles_labels()
    ax.legend(h1 + h2, l1 + l2, fontsize=8, loc="upper right")
    ax = axes[1]
    ax.bar(x - 0.2, w_bull * 100.0, 0.4, color="#2ca25f", label="Bullish (SLSQP)")
    ax.bar(x + 0.2, w_bear * 100.0, 0.4, color="#de2d26", alpha=0.8, label="Bearish (equal weight)")
    ax.axhline(cap * 100.0, color="black", linestyle="--", linewidth=1.0, label="30% cap")
    ax.set_xticks(x); ax.set_xticklabels(names)
    ax.set_ylabel("Weight (%)"); ax.set_ylim(0, 34)
    ax.set_title("Allocation: concentrate (Bullish) vs diversify (Bearish)")
    ax.legend(fontsize=8, loc="upper right")
    fig.suptitle("Toy mini-allocation on four stocks", y=1.0)
    fig.tight_layout()
    fig.savefig(cfg.img_dir / "toy_allocation.png", dpi=180)
    plt.show()
    return {"names": names, "H": H, "mu": mu, "w_bull": w_bull, "w_bear": w_bear}


toy_table = toy_entropy_example(cfg)
toy_alloc = toy_allocation_example(cfg)
print("Toy allocation (Bullish SLSQP):",
      {k: f"{w:.3f}" for k, w in zip(toy_alloc["names"], toy_alloc["w_bull"])})
toy_table


## 5. Rolling entropy signal

Per-ticker rolling Shannon entropy over `lookback_days = 126` days, shifted one day
(no look-ahead), together with the rolling mean and volatility. The expected-return
input is the **plain rolling mean** (`predicted_return = rolling_mean`); entropy is used
only for the regime, never to weight the prediction.

In [ ]:
def build_entropy_signal(cfg, returns, active):
    """Rolling per-ticker signal on the past window [t-W, t-1] (shift(1) -> no
    look-ahead): normalized entropy, mean and volatility. The expected-return input is
    the PLAIN rolling mean; entropy is used only for the regime, never to weight it.
    In: cfg, returns, active. Out: (rolling_entropy, expected_return=rolling_mean,
    rolling_vol, features long-form). O(T*N*W) (rolling entropy dominates)."""
    active_aligned = active.reindex(returns.index).fillna(False).astype(bool)
    roll = returns.rolling(cfg.lookback_days, min_periods=cfg.min_periods)
    rolling_entropy = roll.apply(lambda x: shannon_entropy(x, cfg.entropy_bins), raw=True).shift(1)
    rolling_mean = roll.mean().shift(1)
    rolling_vol = roll.std().shift(1)
    predicted_return = rolling_mean                 # mu_hat = plain rolling mean (no entropy weighting)

    rolling_entropy.to_csv(cfg.output_dir / "rolling_normalized_entropy_matrix.csv", index_label="Date")
    rolling_mean.to_csv(cfg.output_dir / "rolling_mean_return_matrix.csv", index_label="Date")
    rolling_vol.to_csv(cfg.output_dir / "rolling_volatility_matrix.csv", index_label="Date")

    def long(matrix, name):
        return (matrix.where(active_aligned).stack().rename(name)
                .reset_index().rename(columns={"level_1": "Ticker"}))
    features = (long(rolling_entropy, "NormalizedEntropy")
                .merge(long(rolling_mean, "RollingMean"), on=["Date", "Ticker"], how="outer")
                .merge(long(rolling_vol, "RollingVolatility"), on=["Date", "Ticker"], how="outer"))
    features["DisplayTicker"] = features["Ticker"].map(display_ticker)
    features = features.sort_values(["Date", "Ticker"])
    features.to_csv(cfg.output_dir / "entropy_features.csv", index=False)

    params = features.groupby("Date")[["NormalizedEntropy", "RollingMean", "RollingVolatility"]].mean().dropna(how="all")
    params["RollingMeanAnnualized"] = params["RollingMean"] * cfg.trading_days
    params["RollingVolatilityAnnualized"] = params["RollingVolatility"] * math.sqrt(cfg.trading_days)
    params.to_csv(cfg.output_dir / "rolling_parameter_timeseries.csv", index_label="Date")
    return rolling_entropy, predicted_return, rolling_vol, features


def plot_entropy_overview(cfg, features):
    """Fig 1: cross-sectional average rolling entropy over time. Fig 2: the rolling
    parameters that feed the allocation (annualized rolling mean and volatility).
    In: cfg, features. Out: none (saves 2 PNGs). O(T)."""
    if features.empty:
        return
    by_date = features.groupby("Date")["NormalizedEntropy"].mean()
    fig, ax = plt.subplots(figsize=(9.5, 4.8))
    ax.plot(by_date.index, by_date, label="Average normalized entropy", linewidth=1.2)
    ax.axhline(0.5, color="tab:gray", linestyle="--", linewidth=0.8, label="0.5 reference")
    ax.set_title("Rolling entropy in the active top-10 universe")
    ax.set_ylabel("Cross-sectional average entropy")
    ax.set_ylim(0.4, 1.0)   # zoom: the average entropy lives in ~0.55-0.85, so 0-1 wastes space
    ax.grid(alpha=0.25)
    ax.legend()
    fig.tight_layout()
    fig.savefig(cfg.img_dir / "entropy_overview.png", dpi=180)
    plt.show()

    params = pd.read_csv(cfg.output_dir / "rolling_parameter_timeseries.csv", parse_dates=["Date"]).set_index("Date")
    if params.empty:
        return
    fig, axes = plt.subplots(2, 1, figsize=(10.5, 6.6), sharex=True)
    axes[0].plot(params.index, params["RollingMeanAnnualized"], label="Average rolling mean, annualized", linewidth=1.0)
    axes[0].axhline(0.0, color="black", linewidth=0.8)
    axes[0].set_ylabel("Annualized return")
    axes[0].grid(alpha=0.25)
    axes[0].legend(fontsize=8)
    axes[1].plot(params.index, params["RollingVolatilityAnnualized"],
                 label="Average rolling volatility, annualized", linewidth=1.0, color="tab:red")
    axes[1].set_ylabel("Annualized volatility")
    axes[1].set_xlabel("Date")
    axes[1].grid(alpha=0.25)
    axes[1].legend(fontsize=8)
    fig.suptitle("Rolling parameters used by the allocation (expected return = rolling mean)", y=0.99)
    fig.tight_layout()
    fig.savefig(cfg.img_dir / "rolling_signal_parameters.png", dpi=180)
    plt.show()


print("[SIGNAL] Computing rolling entropy and the rolling-mean expected return...")
rolling_entropy, predicted_return, rolling_vol, features = build_entropy_signal(cfg, returns, active)
plot_entropy_overview(cfg, features)
print(f"Entropy matrix: {rolling_entropy.shape}; feature rows: {len(features):,}")


## 6. Market regimes from entropy alone

Each day is Bullish / Neutral / Bearish purely from where the active-universe average
entropy sits relative to its **rolling, causal** 33rd/67th percentile bands (504-day
window, `.shift(1)`): low entropy → Bullish, high entropy → Bearish, middle → Neutral.
Returns/drawdown/volatility are reported per regime but do not drive the rule.

This cell also produces the requested **rolling-quantile image** and the **transition
matrix** (table + heatmap).

In [ ]:
def classify_market_phase(row: pd.Series, cfg: EntropyConfig) -> str:
    """Label one day Bullish/Neutral/Bearish from the active-universe average entropy
    vs its rolling causal 33/67 bands (entropy alone; returns/drawdown/vol excluded).
    In: row with entropy + band columns. Out: phase string. O(1)."""
    average_entropy = row["AverageNormalizedEntropy"]
    entropy_low = row["RollingEntropyLow"]
    entropy_high = row["RollingEntropyHigh"]
    if not (np.isfinite(average_entropy) and np.isfinite(entropy_low) and np.isfinite(entropy_high)):
        return "Neutral"
    if average_entropy <= entropy_low:
        return "Bullish"   # low disorder / ordered market
    if average_entropy >= entropy_high:
        return "Bearish"   # high disorder / crisis-like market
    return "Neutral"


def estimate_market_phases(
    cfg: EntropyConfig,
    returns: pd.DataFrame,
    active: pd.DataFrame,
    rolling_entropy: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Label every day's regime from the top-10 investable-universe average entropy
    vs its rolling causal 33/67 bands (return/vol/drawdown kept only as descriptive
    diagnostics), and build the transition matrix and per-phase summary.
    In: cfg, returns, active, rolling_entropy. Out: (phases, transition_matrix,
    transition_counts, phase_summary). O(T*W) on a single proxy series + O(T) labels."""
    active_aligned = active.reindex(returns.index).fillna(False).astype(bool)
    market_proxy = returns.where(active_aligned).mean(axis=1, skipna=True)
    market_proxy = market_proxy.loc[
        (market_proxy.index >= pd.Timestamp(cfg.warmup_start))
        & (market_proxy.index <= pd.Timestamp(cfg.analysis_end))
    ]
    rolling_return = (
        (1.0 + market_proxy).rolling(cfg.phase_lookback_days, min_periods=cfg.min_periods)
        .apply(lambda x: float(np.prod(x) - 1.0), raw=True).shift(1)
    )
    rolling_volatility = market_proxy.rolling(cfg.phase_lookback_days, min_periods=cfg.min_periods).std().shift(1)
    rolling_volatility = rolling_volatility * math.sqrt(cfg.trading_days)
    market_wealth = (1.0 + market_proxy.fillna(0.0)).cumprod()
    prior_wealth = market_wealth.shift(1)
    rolling_peak = prior_wealth.rolling(cfg.phase_lookback_days, min_periods=cfg.min_periods).max()
    rolling_drawdown = prior_wealth / rolling_peak - 1.0
    average_entropy = rolling_entropy.reindex_like(returns).where(active_aligned).mean(axis=1, skipna=True)
    average_entropy = average_entropy.reindex(market_proxy.index)
    # Rolling, causal 33/67 entropy bands; .shift(1) keeps day t free of its own
    # entropy, and the cutoffs are local to the last phase_entropy_window days.
    entropy_low = average_entropy.rolling(
        cfg.phase_entropy_window, min_periods=cfg.phase_entropy_min_periods
    ).quantile(cfg.phase_entropy_lower_quantile).shift(1)
    entropy_high = average_entropy.rolling(
        cfg.phase_entropy_window, min_periods=cfg.phase_entropy_min_periods
    ).quantile(cfg.phase_entropy_upper_quantile).shift(1)

    phases = pd.DataFrame({
        "Date": market_proxy.index,
        "MarketProxyReturn": market_proxy.to_numpy(dtype=float),
        "MarketProxyWealth": market_wealth.to_numpy(dtype=float),
        "RollingReturn": rolling_return.to_numpy(dtype=float),
        "RollingVolatility": rolling_volatility.to_numpy(dtype=float),
        "RollingDrawdown": rolling_drawdown.to_numpy(dtype=float),
        "AverageNormalizedEntropy": average_entropy.to_numpy(dtype=float),
        "RollingEntropyLow": entropy_low.to_numpy(dtype=float),
        "RollingEntropyHigh": entropy_high.to_numpy(dtype=float),
    })
    phases = phases[
        (phases["Date"] >= pd.Timestamp(cfg.analysis_start))
        & (phases["Date"] <= pd.Timestamp(cfg.analysis_end))
        & phases["MarketProxyReturn"].notna()
    ].copy()
    phases["MarketPhase"] = phases.apply(lambda row: classify_market_phase(row, cfg), axis=1)
    phases.to_csv(cfg.output_dir / "market_phases.csv", index=False)

    next_phase = phases["MarketPhase"].shift(-1)
    transitions = pd.DataFrame({"FromPhase": phases["MarketPhase"], "ToPhase": next_phase}).dropna()
    transition_counts = pd.crosstab(transitions["FromPhase"], transitions["ToPhase"])
    transition_counts = transition_counts.reindex(index=MARKET_PHASE_ORDER, columns=MARKET_PHASE_ORDER, fill_value=0)
    transition_matrix = transition_counts.div(transition_counts.sum(axis=1).replace(0, np.nan), axis=0)
    transition_counts.to_csv(cfg.output_dir / "market_phase_transition_counts.csv")
    transition_matrix.to_csv(cfg.output_dir / "market_phase_transition_matrix.csv")

    summary_rows = []
    total_days = max(len(phases), 1)
    for phase in MARKET_PHASE_ORDER:
        block = phases[phases["MarketPhase"] == phase]
        if block.empty:
            continue
        mean_daily_return = float(block["MarketProxyReturn"].mean())
        summary_rows.append({
            "MarketPhase": phase,
            "Days": int(len(block)),
            "ShareOfSample": len(block) / total_days,
            "MeanDailyReturn": mean_daily_return,
            "AnnualizedMeanReturn": (1.0 + mean_daily_return) ** cfg.trading_days - 1.0,
            "MeanRollingReturn": float(block["RollingReturn"].mean()),
            "MeanAverageEntropy": float(block["AverageNormalizedEntropy"].mean()),
            "MeanRollingVolatility": float(block["RollingVolatility"].mean()),
            "MeanRollingDrawdown": float(block["RollingDrawdown"].mean()),
        })
    phase_summary = pd.DataFrame(summary_rows)
    phase_summary.to_csv(cfg.output_dir / "market_phase_summary.csv", index=False)
    return phases, transition_matrix, transition_counts, phase_summary


def shade_market_phases(ax: plt.Axes, market_phases: pd.DataFrame | None) -> list[Patch]:
    """Shade an axis background by regime and return legend patches.
    In: axes, phases. Out: list of Patch handles. O(T)."""
    if market_phases is None or market_phases.empty:
        return []
    phases_local = market_phases[["Date", "MarketPhase"]].copy()
    phases_local["Date"] = pd.to_datetime(phases_local["Date"])
    phases_local = phases_local.dropna().sort_values("Date")
    if phases_local.empty:
        return []
    segment_id = phases_local["MarketPhase"].ne(phases_local["MarketPhase"].shift()).cumsum()
    for _, block in phases_local.groupby(segment_id):
        phase = str(block["MarketPhase"].iloc[0])
        color = MARKET_PHASE_COLORS.get(phase, "#bdbdbd")
        start = block["Date"].iloc[0]
        end = block["Date"].iloc[-1] + pd.Timedelta(days=1)
        ax.axvspan(start, end, color=color, alpha=0.11, linewidth=0, zorder=0)
    return [
        Patch(facecolor=MARKET_PHASE_COLORS[phase], alpha=0.20, label=phase)
        for phase in MARKET_PHASE_ORDER
        if phase in set(phases_local["MarketPhase"])
    ]


def plot_market_phase_diagnostics(cfg: EntropyConfig, market_phases: pd.DataFrame) -> None:
    """Three stacked panels: 126-day rolling return (descriptive), the average entropy
    with its rolling 33/67 percentile bands (zoomed to 0.4-1.0 for readability, with the
    Neutral band shaded), and the daily regime label. O(T)."""
    if market_phases.empty:
        return
    phases_local = market_phases.copy()
    phases_local["Date"] = pd.to_datetime(phases_local["Date"])
    fig, axes = plt.subplots(3, 1, figsize=(10.5, 7.2), sharex=True,
                             gridspec_kw={"height_ratios": [1.0, 1.4, 0.3]})
    for ax in axes[:2]:
        shade_market_phases(ax, phases_local)
    axes[0].plot(phases_local["Date"], phases_local["RollingReturn"], linewidth=1.0, color="black")
    axes[0].axhline(0.0, color="tab:gray", linestyle="--", linewidth=0.8)
    axes[0].set_ylabel("126d return (descriptive)")
    axes[0].grid(alpha=0.25)
    axes[1].fill_between(phases_local["Date"], phases_local["RollingEntropyLow"],
                         phases_local["RollingEntropyHigh"], color="tab:gray", alpha=0.14,
                         label="Neutral band")
    axes[1].plot(phases_local["Date"], phases_local["AverageNormalizedEntropy"],
                 linewidth=1.1, color="black", label="Average entropy")
    axes[1].plot(phases_local["Date"], phases_local["RollingEntropyLow"], linewidth=0.9,
                 color=MARKET_PHASE_COLORS["Bullish"], linestyle="--", label="Rolling 33% (Bullish at/below)")
    axes[1].plot(phases_local["Date"], phases_local["RollingEntropyHigh"], linewidth=0.9,
                 color=MARKET_PHASE_COLORS["Bearish"], linestyle="--", label="Rolling 67% (Bearish at/above)")
    axes[1].set_ylim(0.4, 1.0)
    axes[1].set_ylabel("Normalized entropy")
    axes[1].grid(alpha=0.25)
    axes[1].legend(fontsize=8, loc="lower left", ncol=2)
    phase_numbers = {phase: i for i, phase in enumerate(MARKET_PHASE_ORDER)}
    colors = [MARKET_PHASE_COLORS.get(phase, "#bdbdbd") for phase in phases_local["MarketPhase"]]
    axes[2].scatter(phases_local["Date"], [phase_numbers.get(p, 0) for p in phases_local["MarketPhase"]],
                    c=colors, s=4, marker="s")
    axes[2].set_yticks(list(phase_numbers.values()))
    axes[2].set_yticklabels(MARKET_PHASE_ORDER, fontsize=7)
    axes[2].set_xlabel("Date")
    axes[2].grid(alpha=0.15, axis="x")
    handles = [Patch(facecolor=MARKET_PHASE_COLORS[p], alpha=0.35, label=p) for p in MARKET_PHASE_ORDER]
    axes[0].legend(handles=handles, ncol=3, fontsize=8, loc="upper left")
    fig.suptitle("Market phase diagnostics from rolling entropy quantiles", y=0.99)
    fig.tight_layout()
    fig.savefig(cfg.img_dir / "market_phase_diagnostics.png", dpi=180)
    plt.show()


def plot_transition_matrix(cfg: EntropyConfig, transition_matrix: pd.DataFrame) -> None:
    """Heatmap of the entropy regime transition matrix. O(1)."""
    if transition_matrix is None or transition_matrix.empty:
        return
    M = transition_matrix.reindex(index=MARKET_PHASE_ORDER, columns=MARKET_PHASE_ORDER)
    values = M.to_numpy(dtype=float)
    fig, ax = plt.subplots(figsize=(5.4, 4.6))
    im = ax.imshow(values, cmap="Blues", vmin=0.0, vmax=1.0)
    ax.set_xticks(range(3)); ax.set_xticklabels(MARKET_PHASE_ORDER)
    ax.set_yticks(range(3)); ax.set_yticklabels(MARKET_PHASE_ORDER)
    ax.set_xlabel("To phase (t+1)")
    ax.set_ylabel("From phase (t)")
    for i in range(3):
        for j in range(3):
            v = values[i, j]
            if np.isfinite(v):
                ax.text(j, i, f"{v:.4f}", ha="center", va="center",
                        color="white" if v >= 0.6 else "black", fontsize=9)
    ax.set_title("Entropy regime transition matrix")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    fig.savefig(cfg.img_dir / "market_phase_transition_heatmap.png", dpi=180)
    plt.show()


print("[REGIME] Estimating market phases...")
market_phases, transition_matrix, transition_counts, phase_summary = estimate_market_phases(cfg, returns, active, rolling_entropy)
plot_market_phase_diagnostics(cfg, market_phases)
plot_transition_matrix(cfg, transition_matrix)
phase_summary


## 7. Walk-forward allocation: SLSQP in Bullish/Neutral, equal weight in Bearish

In the Bullish/Neutral regimes, at each date solve
$\min_w \tfrac{\lambda}{2} w^\top\Sigma w - \mu^\top w$ s.t. $\sum w = 1$,
$0 \le w_i \le c(\text{phase}_t)$, with $c=\{$Bull 0.30, Neutral 0.15$\}$ and $\mu$
the rolling mean. In the Bearish regime the optimizer is skipped and the book is equal
weight ($1/n$, the $c=0.10$ outcome at $n=10$). The QP is solved by
`scipy.optimize.minimize(SLSQP)`; the loop is hand-coded. A naive max-Sharpe baseline
($w_i \propto \max(\text{Sharpe}_i,0)$, $O(N)$) is also backtested for comparison.

In [ ]:
def solve_mean_variance_slsqp(
    mu: np.ndarray,
    covariance: np.ndarray,
    lower_bounds: np.ndarray,
    upper_bounds: np.ndarray,
    risk_aversion: float,
) -> tuple[np.ndarray, bool, str]:
    """Long-only mean-variance QP solved by the library SLSQP routine: minimize
    0.5*risk_aversion*w'Sigma w - mu'w s.t. sum(w)=1, lower<=w<=upper. Sigma is
    symmetrized + jittered so the QP is strictly convex (optimum independent of the
    start, here the always-feasible equal weight). In: mu, cov, bounds, lambda.
    Out: (weights, success, message). O(W*K^2 + I*K^3) per call."""
    n_assets = len(mu)
    lower = np.asarray(lower_bounds, dtype=float)
    upper = np.asarray(upper_bounds, dtype=float)
    x0 = np.full(n_assets, 1.0 / n_assets)   # equal weight is feasible: lower=0 and cap >= 1/n
    cov = np.nan_to_num(np.asarray(covariance, dtype=float), nan=0.0, posinf=0.0, neginf=0.0)
    cov = 0.5 * (cov + cov.T) + np.eye(n_assets) * 1e-8
    mu = np.nan_to_num(np.asarray(mu, dtype=float), nan=0.0, posinf=0.0, neginf=0.0)

    def objective(w):
        return 0.5 * risk_aversion * float(w @ cov @ w) - float(mu @ w)

    def objective_grad(w):
        return risk_aversion * (cov @ w) - mu

    constraints = [{"type": "eq", "fun": lambda w: np.sum(w) - 1.0, "jac": lambda w: np.ones_like(w)}]
    result = minimize(
        objective, x0, method="SLSQP", jac=objective_grad,
        bounds=list(zip(lower, upper)), constraints=constraints,
        options={"ftol": 1e-10, "maxiter": 120, "disp": False},
    )
    if not result.success or not np.all(np.isfinite(result.x)):
        return x0, False, str(result.message)
    return result.x, True, "ok"


def feasible_max_weight(n_active: int, max_w: float) -> float:
    """Relax the cap to 1/n when n*max_w < 1, else sum(w)=1 is infeasible. O(1)."""
    if n_active <= 0:
        return max_w
    return max(max_w, 1.0 / n_active)


def run_portfolio_optimization(cfg, returns, active, predicted_return, market_phases):
    """Daily walk-forward allocation over the active top-10. Bullish/Neutral solve the
    capped mean-variance QP by SLSQP (cap = regime, min = 0); Bearish skips the solver
    and holds equal weight. mu is the plain rolling mean. Writes weights / returns /
    diagnostics / tilts CSVs. In: cfg, returns, active, predicted_return,
    market_phases. Out: (weights_df, returns_df, diagnostics_df). O(T*(W*K^2 + I*K^3))."""
    tickers = list(returns.columns)
    active = active.reindex(returns.index).fillna(False).astype(bool)
    predicted_return = predicted_return.reindex_like(returns)
    phase_by_date = {}
    if not market_phases.empty:
        pf = market_phases[["Date", "MarketPhase"]].copy()
        pf["Date"] = pd.to_datetime(pf["Date"])
        phase_by_date = dict(zip(pf["Date"], pf["MarketPhase"]))

    weights_rows, returns_rows, diagnostic_rows = [], [], []
    tilt_rows = []
    dates = list(returns.index)
    # Positional numpy views of returns/active to replace the per-ticker .at lookups
    # that dominated the active-set detection (the real cost). mu/realized/cov below
    # stay label-based so every float is produced exactly as before (bit-identical).
    ret_np = returns.to_numpy(dtype=float)
    act_np = active.to_numpy(dtype=bool)
    a_start, a_end = pd.Timestamp(cfg.analysis_start), pd.Timestamp(cfg.analysis_end)
    print("[MVO] Running regime allocation (SLSQP in Bullish/Neutral, equal weight in Bearish)...")
    for position in range(cfg.lookback_days + 1, len(dates)):
        date = dates[position]
        if date < a_start or date > a_end:
            continue
        row_active = act_np[position] & np.isfinite(ret_np[position])
        active_idx = np.nonzero(row_active)[0]
        n_active = active_idx.size
        if n_active == 0:
            continue
        active_tickers = [tickers[j] for j in active_idx]
        # Keep the full active top-10 even if a freshly-promoted name lacks a signal
        # (its mu just becomes 0); pairwise covariance avoids dropping the whole day.
        window = returns.iloc[position - cfg.lookback_days: position][active_tickers]
        if len(window) < cfg.min_periods:
            continue
        covariance = window.cov().to_numpy()
        mu = np.nan_to_num(predicted_return.loc[date, active_tickers].to_numpy(dtype=float), nan=0.0)
        realized = returns.loc[date, active_tickers].to_numpy(dtype=float)

        phase = phase_by_date.get(date, "Neutral")
        nominal_max = REGIME_TO_MAX_WEIGHT.get(phase, 0.15)
        stock_min = 0.0
        stock_max = feasible_max_weight(n_active, nominal_max)
        lower = np.full(n_active, stock_min, dtype=float)
        upper = np.full(n_active, stock_max, dtype=float)

        if phase == "Bearish":
            # Disordered market: diversify. Skip the optimizer and hold equal weight
            # over the active top-10 (identical to the 10% cap outcome at n = 10).
            weights = np.full(n_active, 1.0 / n_active)
            success, message = True, "equal-weight (Bearish)"
        else:
            weights, success, message = solve_mean_variance_slsqp(mu, covariance, lower, upper, cfg.risk_aversion)
            if not np.all(np.isfinite(weights)):
                weights = np.full(n_active, 1.0 / n_active)
                success, message = False, "fallback equal weight"

        portfolio_return = float(np.dot(weights, realized))
        equal_weight_return = float(np.mean(realized))
        weight_sum = float(np.sum(weights))
        lower_residual = np.minimum(0.0, weights - lower)
        upper_residual = np.maximum(0.0, weights - upper)
        residuals = np.r_[weight_sum - 1.0, lower_residual, upper_residual]
        two_inf_error = 2.0 * float(np.max(np.abs(residuals)))
        lower_count = int(np.sum(np.isclose(weights, lower, atol=1e-5)))
        upper_count = int(np.sum(np.isclose(weights, upper, atol=1e-5)))
        equal_weight = 1.0 / n_active

        weight_row = {"Date": date}
        for t in tickers:
            weight_row[t] = 0.0
        for t, w in zip(active_tickers, weights):
            weight_row[t] = float(w)
            tilt_rows.append({"Date": date, "Ticker": t, "DisplayTicker": display_ticker(t),
                              "Weight": float(w), "EqualWeight": equal_weight,
                              "TiltVsEqualWeight": float(w - equal_weight),
                              "PositionClass": ("overweight" if w > equal_weight + 1e-5
                                                else ("underweight" if w < equal_weight - 1e-5 else "neutral"))})

        weights_rows.append(weight_row)
        returns_rows.append({"Date": date, "entropy_mvo": portfolio_return,
                             "equal_weight_top10": equal_weight_return})
        diagnostic_rows.append({
            "Date": date, "MarketPhase": phase, "ActiveCount": n_active,
            "InvestedCount": int(np.sum(np.abs(weights) > 1e-8)),
            "WeightSum": weight_sum, "SumError": abs(weight_sum - 1.0),
            "TwoInfNormConstraintError": two_inf_error,
            "LowerBoundCount": lower_count, "UpperBoundCount": upper_count,
            "LowerBoundShare": lower_count / n_active, "UpperBoundShare": upper_count / n_active,
            "SolverSuccess": bool(success), "SolverMessage": message,
            "NominalMaxWeight": nominal_max,
            "SelectedStockMinWeight": stock_min, "SelectedStockMaxWeight": stock_max,
        })

    weights_df = pd.DataFrame(weights_rows)
    returns_df = pd.DataFrame(returns_rows)
    diagnostics_df = pd.DataFrame(diagnostic_rows)
    tilts_df = pd.DataFrame(tilt_rows)
    weights_df.to_csv(cfg.output_dir / "portfolio_weights.csv", index=False)
    returns_df.to_csv(cfg.output_dir / "portfolio_returns.csv", index=False)
    diagnostics_df.to_csv(cfg.output_dir / "optimization_diagnostics.csv", index=False)
    tilts_df.to_csv(cfg.output_dir / "overweight_underweight_tilts.csv", index=False)
    print(f"[MVO] Optimized days: {len(diagnostics_df)}")
    return weights_df, returns_df, diagnostics_df


def run_naive_sharpe(cfg, returns, active, predicted_return, rolling_vol):
    """Naive O(N) baseline (no optimizer): each day weight the active top-10 in
    proportion to their positive rolling Sharpe (rolling mean / rolling vol, already
    shifted -> causal), equal weight if none is positive; rebalanced daily. Writes
    the naive weights CSV. In: cfg + signal matrices. Out: DataFrame[Date,
    naive_sharpe]. O(T*N)."""
    act = active.reindex(returns.index).fillna(False).astype(bool)
    sharpe = (predicted_return / rolling_vol).reindex_like(returns).where(act)
    pos = sharpe.where(np.isfinite(sharpe)).clip(lower=0.0).fillna(0.0)
    row_sum = pos.sum(axis=1)
    equal = act.div(act.sum(axis=1).replace(0, np.nan), axis=0)        # equal weight over active names
    weights = pos.div(row_sum, axis=0).where(row_sum > 0, equal).fillna(0.0)
    daily = (weights * returns.where(act).fillna(0.0)).sum(axis=1)
    mask = ((returns.index >= pd.Timestamp(cfg.analysis_start))
            & (returns.index <= pd.Timestamp(cfg.analysis_end)) & (act.sum(axis=1) > 0))
    weights[mask].to_csv(cfg.output_dir / "naive_sharpe_weights.csv", index_label="Date")
    return pd.DataFrame({"Date": returns.index[mask], "naive_sharpe": daily[mask].to_numpy(dtype=float)})


weights_df, returns_df, diagnostics = run_portfolio_optimization(cfg, returns, active, predicted_return, market_phases)
returns_df = returns_df.merge(run_naive_sharpe(cfg, returns, active, predicted_return, rolling_vol), on="Date", how="left")
returns_df.to_csv(cfg.output_dir / "portfolio_returns.csv", index=False)
diagnostics[["MarketPhase", "ActiveCount", "InvestedCount", "SelectedStockMaxWeight"]].head()


## 8. Performance and benchmarks

Compared against a **naive max-Sharpe top-10**, **equal-weight top-10**, and the
**S&P 500 total-return buy-and-hold** (no rebalancing), all rebased to 1 at the first
optimized day.

In [ ]:
def max_drawdown(returns_series: pd.Series) -> float:
    """Worst peak-to-trough loss of the wealth curve. In: returns. Out: float<=0. O(T)."""
    clean = returns_series.dropna()
    if clean.empty:
        return np.nan
    wealth = (1.0 + clean).cumprod()
    drawdown = wealth / wealth.cummax() - 1.0
    return float(drawdown.min())


def annualized_return(returns_series: pd.Series, trading_days: int) -> float:
    """Geometric annualized return. In: daily returns. Out: float. O(T)."""
    clean = returns_series.dropna()
    if clean.empty:
        return np.nan
    return float((1.0 + clean).prod() ** (trading_days / len(clean)) - 1.0)


def annualized_volatility(returns_series: pd.Series, trading_days: int) -> float:
    """Annualized standard deviation of daily returns. In: returns. Out: float. O(T)."""
    clean = returns_series.dropna()
    if len(clean) < 2:
        return np.nan
    return float(clean.std(ddof=1) * math.sqrt(trading_days))


def sharpe_ratio(returns_series: pd.Series, trading_days: int) -> float:
    """Annualized return / annualized volatility (rf = 0). In: returns. Out: float. O(T)."""
    ar = annualized_return(returns_series, trading_days)
    av = annualized_volatility(returns_series, trading_days)
    if not av or not np.isfinite(ar) or not np.isfinite(av):
        return np.nan
    return float(ar / av)


STRATEGY_LABELS = {
    "entropy_mvo": "Regime-adaptive entropy MVO",
    "naive_sharpe": "Naive max-Sharpe top-10",
    "equal_weight_top10": "Equal-weight active top-10",
    "sp500_buy_hold": "S&P 500 total-return buy & hold",
}
STRATEGY_ORDER = ["entropy_mvo", "naive_sharpe", "equal_weight_top10", "sp500_buy_hold"]


def load_sp500_returns(cfg: EntropyConfig, dates_index) -> pd.Series:
    """Daily return of the S&P 500 total-return index, aligned to the optimized dates.
    In: cfg, dates. Out: returns Series. O(T)."""
    s = pd.read_csv(cfg.sp500_path)
    dcol = "Date" if "Date" in s.columns else s.columns[0]
    ccol = "Close" if "Close" in s.columns else s.columns[-1]
    s[dcol] = pd.to_datetime(s[dcol])
    series = s.set_index(dcol)[ccol].sort_index().astype(float)
    ret = series.pct_change()
    return ret.reindex(pd.DatetimeIndex(dates_index)).fillna(0.0)


def performance_metrics(cfg: EntropyConfig, returns_df: pd.DataFrame) -> pd.DataFrame:
    """Annualized return/vol/Sharpe, max drawdown and final wealth for every strategy
    column present. In: cfg, returns_df. Out: metrics DataFrame. O(S*T)."""
    rows = []
    for strategy in STRATEGY_ORDER:
        if strategy not in returns_df.columns:
            continue
        series = returns_df[strategy].dropna()
        rows.append({
            "Strategy": strategy,
            "Observations": int(len(series)),
            "AnnualizedReturn": annualized_return(series, cfg.trading_days),
            "AnnualizedVolatility": annualized_volatility(series, cfg.trading_days),
            "SharpeRatio": sharpe_ratio(series, cfg.trading_days),
            "MaxDrawdown": max_drawdown(series),
            "FinalWealth": float((1.0 + series).prod()) if not series.empty else np.nan,
        })
    metrics = pd.DataFrame(rows)
    metrics.to_csv(cfg.output_dir / "performance_metrics.csv", index=False)
    return metrics


def plot_cumulative_returns(cfg: EntropyConfig, returns_df: pd.DataFrame, market_phases: pd.DataFrame) -> None:
    """Growth-of-1 wealth curves for all strategies, shaded by entropy regime.
    In: cfg, returns_df, market_phases. Out: none (saves PNG). O(S*T)."""
    df = returns_df.copy()
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.set_index("Date")
    series_cols = [c for c in STRATEGY_ORDER if c in df.columns]
    wealth = (1.0 + df[series_cols].fillna(0.0)).cumprod()
    styles = {"entropy_mvo": dict(lw=1.5, color="tab:blue"),
              "naive_sharpe": dict(lw=1.1, color="tab:purple", ls=":"),
              "equal_weight_top10": dict(lw=1.2, color="tab:orange"),
              "sp500_buy_hold": dict(lw=1.2, color="tab:green", ls="--")}
    fig, ax = plt.subplots(figsize=(9.8, 5.2))
    phase_handles = shade_market_phases(ax, market_phases)
    for c in series_cols:
        ax.plot(wealth.index, wealth[c], label=STRATEGY_LABELS.get(c, c), zorder=2, **styles.get(c, {}))
    ax.set_title("Cumulative wealth vs benchmarks (entropy-regime background)")
    ax.set_ylabel("Growth of 1")
    ax.grid(alpha=0.25)
    lh, ll = ax.get_legend_handles_labels()
    ax.legend(lh + phase_handles, ll + [h.get_label() for h in phase_handles], fontsize=8, ncol=2)
    fig.tight_layout()
    fig.savefig(cfg.img_dir / "cumulative_returns.png", dpi=180)
    plt.show()


returns_df = returns_df.copy()
returns_df["sp500_buy_hold"] = load_sp500_returns(cfg, pd.to_datetime(returns_df["Date"])).to_numpy()
metrics = performance_metrics(cfg, returns_df)
plot_cumulative_returns(cfg, returns_df, market_phases)
metrics


## 9. Crisis-window analysis

Behaviour through the four stress windows (2008, 2011, COVID, 2022), with the S&P 500
buy-and-hold line added. The per-regime weight cap during each crisis is shown too.

In [ ]:
def add_small_inside_legend(ax: plt.Axes, loc: str = "best") -> None:
    """Attach a compact in-axes legend. In: axes. Out: none. O(1)."""
    handles, labels = ax.get_legend_handles_labels()
    if not handles:
        return
    ax.legend(
        handles, labels, loc=loc, fontsize=6.8, frameon=True, framealpha=0.88,
        borderpad=0.35, labelspacing=0.28, handlelength=1.3, handletextpad=0.45,
    )


def crisis_legend_label(label: str, returns_series: pd.Series, trading_days: int) -> str:
    """Legend label with cumulative return, max drawdown and Sharpe for a crisis window.
    In: label, returns. Out: string. O(T)."""
    clean = returns_series.dropna()
    if clean.empty:
        return label
    cumulative_return = float((1.0 + clean).prod() - 1.0)
    crisis_dd = max_drawdown(clean)
    crisis_s = sharpe_ratio(clean, trading_days)
    sharpe_text = f"{crisis_s:.2f}" if np.isfinite(crisis_s) else "n/a"
    return f"{label} | ret {cumulative_return:+.1%} | max DD {crisis_dd:.1%} | Sharpe {sharpe_text}"


def crisis_performance_table(cfg: EntropyConfig, returns_df: pd.DataFrame, diagnostics: pd.DataFrame) -> pd.DataFrame:
    """Per-crisis performance of the entropy MVO and equal-weight books, with the
    average per-name regime cap. In: cfg, returns_df, diagnostics. Out: table (also
    saved to crisis_performance.csv). O(C*T)."""
    returns_local = returns_df.copy()
    returns_local["Date"] = pd.to_datetime(returns_local["Date"])
    returns_local = returns_local.set_index("Date").sort_index()
    diag_local = diagnostics.copy()
    diag_local["Date"] = pd.to_datetime(diag_local["Date"])
    diag_local = diag_local.set_index("Date").sort_index()
    rows = []
    for name, start, end in CRISIS_WINDOWS:
        s_ts, e_ts = pd.Timestamp(start), pd.Timestamp(end)
        block = returns_local.loc[(returns_local.index >= s_ts) & (returns_local.index <= e_ts)]
        diag_block = diag_local.loc[(diag_local.index >= s_ts) & (diag_local.index <= e_ts)]
        if block.empty:
            continue
        for column, label in [
            ("entropy_mvo", "Regime-adaptive entropy MVO"),
            ("equal_weight_top10", "Equal-weight active universe"),
        ]:
            series = block[column].dropna()
            rows.append({
                "Crisis": name, "Start": start, "End": end, "Strategy": label,
                "Observations": int(len(series)),
                "CumulativeReturn": float((1.0 + series).prod() - 1.0) if not series.empty else np.nan,
                "AnnualizedReturn": annualized_return(series, cfg.trading_days),
                "AnnualizedVolatility": annualized_volatility(series, cfg.trading_days),
                "SharpeRatio": sharpe_ratio(series, cfg.trading_days),
                "MaxDrawdown": max_drawdown(series),
                "AverageSelectedMin": float(diag_block["SelectedStockMinWeight"].mean()) if not diag_block.empty else np.nan,
                "AverageSelectedMax": float(diag_block["SelectedStockMaxWeight"].mean()) if not diag_block.empty else np.nan,
            })
    table = pd.DataFrame(rows)
    table.to_csv(cfg.output_dir / "crisis_performance.csv", index=False)
    return table


def plot_crisis_behavior(cfg, returns_df, diagnostics, market_phases):
    """Rebased wealth per crisis (entropy vs equal-weight vs S&P 500) plus the
    per-regime weight cap. In: cfg, returns_df, diagnostics, phases. Out: 2 PNGs. O(C*T)."""
    import matplotlib.dates as mdates

    def _fmt_dates(ax):
        # Compact, non-overlapping x-axis dates for the short crisis windows.
        loc = mdates.AutoDateLocator(maxticks=4)
        ax.xaxis.set_major_locator(loc)
        ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(loc))
        ax.tick_params(axis="x", labelsize=7, rotation=15)

    returns_local = returns_df.copy()
    returns_local["Date"] = pd.to_datetime(returns_local["Date"])
    returns_local = returns_local.set_index("Date").sort_index()
    diag_local = diagnostics.copy()
    diag_local["Date"] = pd.to_datetime(diag_local["Date"])
    diag_local = diag_local.set_index("Date").sort_index()
    cols = [c for c in ["entropy_mvo", "equal_weight_top10", "sp500_buy_hold"] if c in returns_local.columns]
    labels = {"entropy_mvo": "Entropy MVO", "equal_weight_top10": "Equal weight",
              "sp500_buy_hold": "S&P 500"}

    fig, axes = plt.subplots(2, 2, figsize=(11.5, 7.2))
    for ax, (name, start, end) in zip(axes.ravel(), CRISIS_WINDOWS):
        s_ts, e_ts = pd.Timestamp(start), pd.Timestamp(end)
        block = returns_local.loc[(returns_local.index >= s_ts) & (returns_local.index <= e_ts), cols]
        if block.empty:
            ax.set_visible(False)
            continue
        wealth = (1.0 + block.fillna(0.0)).cumprod()
        wealth = wealth / wealth.iloc[0]
        phase_block = market_phases[(pd.to_datetime(market_phases["Date"]) >= s_ts)
                                    & (pd.to_datetime(market_phases["Date"]) <= e_ts)]
        shade_market_phases(ax, phase_block)
        for c in cols:
            ax.plot(wealth.index, wealth[c],
                    label=crisis_legend_label(labels.get(c, c), block[c], cfg.trading_days),
                    linewidth=1.2, zorder=2)
        ax.set_title(name)
        ax.set_ylabel("Rebased wealth")
        ax.grid(alpha=0.25)
        _fmt_dates(ax)
        add_small_inside_legend(ax)
    fig.suptitle("Crisis-period behaviour, wealth rebased at each crisis start", y=0.99)
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    fig.savefig(cfg.img_dir / "crisis_behavior.png", dpi=180)
    plt.show()

    fig, axes = plt.subplots(2, 2, figsize=(11.5, 7.2), sharey=True)
    for ax, (name, start, end) in zip(axes.ravel(), CRISIS_WINDOWS):
        s_ts, e_ts = pd.Timestamp(start), pd.Timestamp(end)
        block = diag_local.loc[(diag_local.index >= s_ts) & (diag_local.index <= e_ts)]
        if block.empty:
            ax.set_visible(False)
            continue
        ax.step(block.index, block["SelectedStockMaxWeight"], where="post",
                color="tab:red", linewidth=1.3, label="Per-name max weight (regime cap)")
        ax.set_title(name)
        ax.set_ylabel("Weight cap")
        ax.set_ylim(0, 0.32)
        ax.grid(alpha=0.25)
        _fmt_dates(ax)
        add_small_inside_legend(ax)
    fig.suptitle("Per-regime weight cap during crisis windows", y=0.99)
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    fig.savefig(cfg.img_dir / "crisis_bounds.png", dpi=180)
    plt.show()


crisis_table = crisis_performance_table(cfg, returns_df, diagnostics)
plot_crisis_behavior(cfg, returns_df, diagnostics, market_phases)
crisis_table[crisis_table["Strategy"] == "Regime-adaptive entropy MVO"]


## 10. Weight diagnostics

Weight heatmap, weight-sum/constraint check, number of traded names through time
(stays at the active top-10), and the per-regime maximum weight over time
(`dynamic_bounds_over_time.png`: min is flat at 0, max steps through 10/15/30).

In [ ]:
def plot_weights_diagnostics(cfg, weights_df, diagnostics):
    """Weight-exposure heatmap, number of traded names through time (stays at the
    active top-10), and the per-regime weight cap through time. The sum-of-weights and
    constraint-error panels are intentionally omitted: the weights sum to 1 by
    construction (a flat, uninformative line) and the residual is at machine precision.
    O(T*N)."""
    weights_local = weights_df.copy()
    weights_local["Date"] = pd.to_datetime(weights_local["Date"])
    weights_local = weights_local.set_index("Date")
    weight_cols = list(weights_local.columns)
    monthly = weights_local[weight_cols].resample(MONTH_END_FREQ).last().fillna(0.0)
    labels = [display_ticker(c) for c in monthly.columns]
    values = monthly.to_numpy(dtype=float)
    vmin = min(0.0, float(np.nanmin(values)) if values.size else 0.0)
    vmax = max(cfg.max_weight, float(np.nanmax(values)) if values.size else cfg.max_weight)
    cmap = "RdBu_r" if vmin < 0.0 else "viridis"
    fig, ax = plt.subplots(figsize=(10.5, max(5.0, 0.22 * len(labels))))
    im = ax.imshow(monthly.T, aspect="auto", interpolation="nearest", vmin=vmin, vmax=vmax, cmap=cmap)
    ax.set_title("Portfolio weights through time")
    ax.set_yticks(np.arange(len(labels)))
    ax.set_yticklabels(labels, fontsize=7)
    if len(monthly.index) > 0:
        xs = np.linspace(0, len(monthly.index) - 1, min(8, len(monthly.index))).astype(int)
        ax.set_xticks(xs)
        ax.set_xticklabels([monthly.index[i].strftime("%Y-%m") for i in xs], rotation=35, ha="right")
    cbar = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
    cbar.set_label("Weight")
    fig.tight_layout()
    fig.savefig(cfg.img_dir / "weight_exposure_heatmap.png", dpi=180)
    plt.show()

    diag_local = diagnostics.copy()
    diag_local["Date"] = pd.to_datetime(diag_local["Date"])
    fig, ax = plt.subplots(figsize=(9.5, 4.8))
    ax.plot(diag_local["Date"], diag_local["ActiveCount"], label="Active names (top-10)", linewidth=1.2)
    ax.plot(diag_local["Date"], diag_local["InvestedCount"], label="Names with non-zero weight", linewidth=1.1)
    ax.set_title("Number of traded names through time")
    ax.set_ylabel("Count")
    ax.set_ylim(0, 12)
    ax.grid(alpha=0.25)
    ax.legend()
    fig.tight_layout()
    fig.savefig(cfg.img_dir / "traded_names_over_time.png", dpi=180)
    plt.show()

    fig, ax = plt.subplots(figsize=(9.5, 4.8))
    ax.plot(diag_local["Date"], diag_local["SelectedStockMinWeight"], label="Per-name minimum (= 0)", linewidth=1.1)
    ax.plot(diag_local["Date"], diag_local["SelectedStockMaxWeight"], label="Per-name maximum (regime cap)", linewidth=1.3, color="tab:red")
    ax.axhline(0.0, color="black", linewidth=0.8)
    ax.set_title("Per-regime weight cap through time")
    ax.set_ylabel("Weight")
    ax.set_ylim(-0.02, 0.34)
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8, ncol=2)
    fig.tight_layout()
    fig.savefig(cfg.img_dir / "dynamic_bounds_over_time.png", dpi=180)
    plt.show()


def plot_bound_usage_by_phase(cfg, diagnostics):
    """Number of names at the lower bound (weight 0) and at the upper bound (the
    regime cap), split into one panel per regime. The upper cap differs by phase
    (30/15/10%); in Bearish every name sits at the cap (equal weight), so the
    upper-bound count equals the active count. Overwrites bound_usage.png. O(T)."""
    d = diagnostics.copy()
    d["Date"] = pd.to_datetime(d["Date"])
    d = d.set_index("Date").sort_index()
    upper_colors = {"Bullish": "#2ca25f", "Neutral": "#3182bd", "Bearish": "#de2d26"}
    lower_color = "#6a3d9a"
    fig, axes = plt.subplots(3, 1, figsize=(10.5, 8.6), sharex=True)
    for i, (ax, phase) in enumerate(zip(axes, MARKET_PHASE_ORDER)):
        mask = d["MarketPhase"] == phase
        cap = REGIME_TO_MAX_WEIGHT[phase]
        upper = d["UpperBoundCount"].where(mask)
        lower = d["LowerBoundCount"].where(mask)
        uc = upper_colors[phase]
        ax.fill_between(d.index, 0.0, upper, color=uc, alpha=0.22, linewidth=0)
        ax.plot(d.index, upper, color=uc, linewidth=1.2,
                label="At upper bound (w = %d%%)" % int(round(cap * 100)))
        ax.plot(d.index, lower, color=lower_color, linewidth=1.1, label="At lower bound (w = 0)")
        ax.set_title("%s phase  -  upper cap %d%%, lower 0%%" % (phase, int(round(cap * 100))),
                     color=uc, fontweight="bold")
        ax.set_ylabel("Number of names")
        ax.set_ylim(0, 10.6)
        ax.grid(alpha=0.2)
        # Bullish/Neutral: data sits low -> legend top-right; Bearish: every name at the
        # cap (line pinned at 10) -> put the legend at the bottom so it clears the data.
        loc = "lower right" if phase == "Bearish" else "upper right"
        ax.legend(fontsize=8, loc=loc, ncol=2, framealpha=0.9)
    axes[-1].set_xlabel("Date")
    fig.suptitle("Names at the lower / upper weight bound, split by regime", y=0.995, fontsize=12)
    fig.tight_layout(rect=[0, 0, 1, 0.97])
    fig.savefig(cfg.img_dir / "bound_usage.png", dpi=180)
    plt.show()


def plot_weight_feasibility(cfg, diagnostics):
    """Constraint feasibility error through time: twice the infinity norm of the weight
    residual (sum-to-1 plus the bound residuals). It is plotted on its own auto-scaled
    axis so the variation is visible, with the tolerance drawn at DOUBLE the worst
    observed error. The error never leaves machine precision (~1e-15), so every daily
    solution is feasible. Overwrites weight_sum_error.png. O(T)."""
    d = diagnostics.copy()
    d["Date"] = pd.to_datetime(d["Date"])
    err = d["TwoInfNormConstraintError"].to_numpy(dtype=float)
    worst = float(np.nanmax(err)) if err.size else 0.0
    tol = 2.0 * worst if worst > 0 else 1e-12
    fig, ax = plt.subplots(figsize=(10.0, 4.4))
    ax.plot(d["Date"], err, linewidth=0.9, color="tab:blue", label="Constraint error (2 x inf-norm)")
    ax.axhline(tol, color="tab:red", linestyle="--", linewidth=1.1,
               label="Tolerance = 2 x max error (%.1e)" % tol)
    ax.axhline(worst, color="tab:gray", linestyle=":", linewidth=0.9,
               label="Max error (%.1e)" % worst)
    ax.set_ylim(0.0, tol * 1.15)
    ax.set_title("Constraint feasibility error through time (machine precision)")
    ax.set_ylabel("2 x infinity-norm of weight residual")
    ax.set_xlabel("Date")
    ax.grid(alpha=0.25)
    ax.legend(fontsize=8, loc="upper right")
    fig.tight_layout()
    fig.savefig(cfg.img_dir / "weight_sum_error.png", dpi=180)
    plt.show()


plot_weights_diagnostics(cfg, weights_df, diagnostics)
plot_bound_usage_by_phase(cfg, diagnostics)
plot_weight_feasibility(cfg, diagnostics)
print("Weights diagnostics done.")


## 11. Report tables and run summary

In [ ]:
def write_run_summary(cfg, metrics, diagnostics, phase_summary):
    """Write run_summary.md: per-strategy performance, diagnostics and phase counts.
    In: cfg, metrics, diagnostics, phase_summary. Out: none (writes markdown). O(S)."""
    def row(name):
        sub = metrics[metrics["Strategy"] == name]
        return sub.iloc[0] if not sub.empty else None
    labels = [("entropy_mvo", "Entropy MVO"), ("naive_sharpe", "Naive max-Sharpe"),
              ("equal_weight_top10", "Equal-weight top-10"), ("sp500_buy_hold", "S&P 500 buy & hold")]
    lines = ["# Entropy Run Summary", "",
             f"Dataset: `{cfg.dataset_path.name}` | Universe: `{cfg.universe_path.name}` | Benchmark: `{cfg.sp500_path.name}`",
             f"Analysis period: {cfg.analysis_start} to {cfg.analysis_end}",
             f"Optimized trading days: {len(diagnostics)}", "",
             "## Performance (final wealth, growth of 1)", ""]
    for key, label in labels:
        r = row(key)
        if r is not None:
            lines += [f"- {label}: {r['FinalWealth']:.3f} "
                      f"(ann {r['AnnualizedReturn']:.2%}, Sharpe {r['SharpeRatio']:.3f}, "
                      f"maxDD {r['MaxDrawdown']:.2%})"]
    lines += ["", "## Diagnostics", "",
              f"- Mean active names: {diagnostics['ActiveCount'].mean():.2f} "
              f"(max {int(diagnostics['ActiveCount'].max())})",
              f"- Mean invested names: {diagnostics['InvestedCount'].mean():.2f}",
              f"- Solver success share: {diagnostics['SolverSuccess'].mean():.2%}",
              f"- Max constraint error: {diagnostics['TwoInfNormConstraintError'].max():.3e}",
              "", "## Market phase counts", ""]
    for _, r in phase_summary.iterrows():
        lines.append(f"- {r['MarketPhase']}: {int(r['Days'])} days ({r['ShareOfSample']:.1%})")
    lines += ["", "## Output", "", f"- CSV: `{cfg.output_dir}`", f"- Figures: `{cfg.img_dir}`"]
    (cfg.output_dir / "run_summary.md").write_text("\n".join(lines), encoding="utf-8")


write_run_summary(cfg, metrics, diagnostics, phase_summary)

n_img = len(list(cfg.img_dir.glob("*.png")))
n_csv = len(list(cfg.output_dir.glob("*.csv")))
print("=" * 64)
for _, r in metrics.iterrows():
    print(f"{STRATEGY_LABELS.get(r['Strategy'], r['Strategy']):32s} | "
          f"final wealth {r['FinalWealth']:.3f} | ann {r['AnnualizedReturn']:.2%} | "
          f"Sharpe {r['SharpeRatio']:.2f} | maxDD {r['MaxDrawdown']:.2%}")
print("=" * 64)
print(f"Figures: {n_img} PNG in {cfg.img_dir}")
print(f"Tables:  {n_csv} CSV in {cfg.output_dir}")
print("Done.")
